# Appliance Energy Consumption Prediction

This project explores whether indoor environmental measurements and time features can be used to estimate household appliance energy consumption. The target is `Appliances`, measured in watt-hours in the source dataset.

The notebook uses a chronological holdout so that the test set represents later observations than the training set. Preprocessing is fitted only on the training data through scikit-learn pipelines.


## 1. Setup


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


## 2. Load the dataset

Download `energydata_complete.csv` from the [UCI Appliances Energy Prediction dataset](https://archive.ics.uci.edu/dataset/374/appliances+energy+prediction) and place it in `data/raw/`. The dataset is not committed to this repository.


In [ ]:
DATA_PATH = Path("data/raw/energydata_complete.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Download it from UCI and place it there."
    )

df = pd.read_csv(DATA_PATH, parse_dates=["date"])
df.head()


In [ ]:
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
display(df.dtypes)
display(df.isna().sum().sort_values(ascending=False).head(10))


## 3. Prepare features

Extract calendar features from the timestamp, then remove the original timestamp. The target and timestamp are excluded from the input features.


In [ ]:
TARGET = "Appliances"

df = df.sort_values("date").reset_index(drop=True)
df["hour"] = df["date"].dt.hour
df["day_of_week"] = df["date"].dt.dayofweek
df["month"] = df["date"].dt.month

X = df.drop(columns=[TARGET, "date"])
y = df[TARGET]

split_at = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_at], X.iloc[split_at:]
y_train, y_test = y.iloc[:split_at], y.iloc[split_at:]

print(f"Training observations: {len(X_train):,}")
print(f"Test observations: {len(X_test):,}")
print(f"Training period: {df['date'].iloc[0]} to {df['date'].iloc[split_at - 1]}")
print(f"Test period: {df['date'].iloc[split_at]} to {df['date'].iloc[-1]}")


## 4. Train baseline and regression models

The baseline predicts the mean appliance consumption observed in the training set. Linear regression provides a simple reference model, while the random forest can capture non-linear relationships. Imputation and scaling are fitted inside each pipeline using training data only.


In [ ]:
models = {
    "Mean baseline": DummyRegressor(strategy="mean"),
    "Linear regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LinearRegression()),
    ]),
    "Random forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=200,
            random_state=42,
            n_jobs=-1,
        )),
    ]),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    results.append({
        "model": name,
        "MAE": mean_absolute_error(y_test, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
        "R2": r2_score(y_test, predictions),
    })

results_df = pd.DataFrame(results).sort_values("RMSE")
results_df


## 5. Interpret the evaluation

Compare each model with the mean baseline. MAE and RMSE are reported in the target's original unit (Wh); lower values are better. R² describes the proportion of test-set variance explained by the model. The chronological split means these results measure performance on a later period, rather than a random sample of the same period.

No performance claim is made in this repository until the notebook has been run against the source dataset in a clean environment.
